In [0]:
raw = "/Volumes/workspace/default/kkbox_raw"
members = spark.read.csv(raw + "/members_v3.csv", header=True, inferSchema=True)
members.count()

6769473

In [0]:
train = spark.read.csv(raw + "/train_v2.csv", header=True, inferSchema=True)
train.count()

970960

In [0]:
transactions = spark.read.csv(raw + "/transactions_v2.csv", header=True, inferSchema=True)
transactions.count()

1431009

In [0]:
logs = spark.read.csv(raw + "/user_logs_v2.csv", header=True, inferSchema=True)
logs.count()

18396362

In [0]:
members.show(5)

+--------------------+----+---+------+--------------+----------------------+
|                msno|city| bd|gender|registered_via|registration_init_time|
+--------------------+----+---+------+--------------+----------------------+
|Rb9UwLQTrxzBVwCB6...|   1|  0|  NULL|            11|              20110911|
|+tJonkh+O1CA796Fm...|   1|  0|  NULL|             7|              20110914|
|cV358ssn7a0f7jZOw...|   1|  0|  NULL|            11|              20110915|
|9bzDeJP6sQodK73K5...|   1|  0|  NULL|            11|              20110915|
|WFLY3s7z4EZsieHCt...|   6| 32|female|             9|              20110915|
+--------------------+----+---+------+--------------+----------------------+
only showing top 5 rows


In [0]:
members.describe("bd").show()

+-------+------------------+
|summary|                bd|
+-------+------------------+
|  count|           6769473|
|   mean| 9.795794295951842|
| stddev|17.925899719010875|
|    min|             -7168|
|    max|              2016|
+-------+------------------+



In [0]:
members.filter(members["bd"] == 0).count()

4540215

In [0]:
from pyspark.sql.functions import when, col

members = members.withColumn("age", when((col("bd") >= 10) & (col("bd") <= 80), col("bd")))

In [0]:
members.select("bd", "age").show(10)

+---+----+
| bd| age|
+---+----+
|  0|NULL|
|  0|NULL|
|  0|NULL|
|  0|NULL|
| 32|  32|
| 30|  30|
|  0|NULL|
| 34|  34|
| 19|  19|
| 63|  63|
+---+----+
only showing top 10 rows


In [0]:
members = members.fillna({"gender": "unknown"})

In [0]:
from pyspark.sql.functions import to_date

members = members.withColumn("signup_date", to_date(col("registration_init_time").cast("string"), "yyyyMMdd"))

In [0]:
members.select("gender", "registration_init_time", "signup_date").show(5)

+-------+----------------------+-----------+
| gender|registration_init_time|signup_date|
+-------+----------------------+-----------+
|unknown|              20110911| 2011-09-11|
|unknown|              20110914| 2011-09-14|
|unknown|              20110915| 2011-09-15|
|unknown|              20110915| 2011-09-15|
| female|              20110915| 2011-09-15|
+-------+----------------------+-----------+
only showing top 5 rows


#PAYMENTS FILE(TRANSACTIONS)

In [0]:
transactions.show(5)

+--------------------+-----------------+-----------------+---------------+------------------+-------------+----------------+----------------------+---------+
|                msno|payment_method_id|payment_plan_days|plan_list_price|actual_amount_paid|is_auto_renew|transaction_date|membership_expire_date|is_cancel|
+--------------------+-----------------+-----------------+---------------+------------------+-------------+----------------+----------------------+---------+
|++6eU4LsQ3UQ20ILS...|               32|               90|            298|               298|            0|        20170131|              20170504|        0|
|++lvGPJOinuin/8es...|               41|               30|            149|               149|            1|        20150809|              20190412|        0|
|+/GXNtXWQVfKrEDqY...|               36|               30|            180|               180|            1|        20170303|              20170422|        0|
|+/w1UrZwyka4C9oNH...|               36|            

In [0]:
transactions = spark.read.csv(raw + "/transactions_v2.csv", header=True, inferSchema=True)
transactions = transactions.withColumn("pay_date", to_date(col("transaction_date").cast("string"), "yyyyMMdd"))
transactions = transactions.withColumn("expire_date", to_date(col("membership_expire_date").cast("string"), "yyyyMMdd"))
transactions = transactions.withColumn("discount", col("plan_list_price") - col("actual_amount_paid"))

In [0]:
transactions.select("pay_date", "expire_date", "discount").show(5)

+----------+-----------+--------+
|  pay_date|expire_date|discount|
+----------+-----------+--------+
|2017-01-31| 2017-05-04|       0|
|2015-08-09| 2019-04-12|       0|
|2017-03-03| 2017-04-22|       0|
|2017-03-29| 2017-03-31|       0|
|2017-03-23| 2017-04-23|       0|
+----------+-----------+--------+
only showing top 5 rows


In [0]:
logs = spark.read.csv(raw + "/user_logs_v2.csv", header=True, inferSchema=True)
logs = logs.withColumn("listen_date", to_date(col("date").cast("string"), "yyyyMMdd"))
logs = logs.filter((col("total_secs") >= 0) & (col("total_secs") <= 86400))

In [0]:
logs.count()

18392162

In [0]:
train.groupBy("is_churn").count().show()

+--------+------+
|is_churn| count|
+--------+------+
|       1| 87330|
|       0|883630|
+--------+------+



#summarize listening per customer. This turns 18 million daily rows into one row per customer.

In [0]:
import pyspark.sql.functions as F

listening = logs.groupBy("msno").agg(
    F.sum("total_secs").alias("total_secs"),
    F.countDistinct("listen_date").alias("active_days"),
    F.sum("num_unq").alias("unique_songs")
)

In [0]:
print(listening.count())
listening.show(5)

1103873
+--------------------+------------------+-----------+------------+
|                msno|        total_secs|active_days|unique_songs|
+--------------------+------------------+-----------+------------+
|cFBEOSfH9NRyMihuz...|110674.00399999999|         30|         650|
|bJ62eAJApxS84sZBF...|50024.202999999994|         19|         236|
|dxunjR64ym+3VM65o...|        343806.809|         31|        2117|
|Didy90bC9/ougkkW3...|        137765.439|         30|         588|
|uP7z1X+bp67MJjA5Y...|        592752.925|         31|        2171|
+--------------------+------------------+-----------+------------+
only showing top 5 rows


#summarize payments per customer.

In [0]:
payments = transactions.groupBy("msno").agg(
    F.count("*").alias("num_payments"),
    F.sum("actual_amount_paid").alias("total_paid"),
    F.max("is_cancel").alias("ever_cancelled"),
    F.max("is_auto_renew").alias("auto_renew"),
    F.avg("discount").alias("avg_discount"),
    F.avg("payment_plan_days").alias("plan_days")
)

In [0]:
print(payments.count())
payments.show(5)

1197050
+--------------------+------------+----------+--------------+----------+------------+---------+
|                msno|num_payments|total_paid|ever_cancelled|auto_renew|avg_discount|plan_days|
+--------------------+------------+----------+--------------+----------+------------+---------+
|+50G1jyzn4OzqalwB...|           1|      1788|             0|         0|         0.0|    410.0|
|+OoYcdCxfGKVHVYS3...|           1|        99|             0|         1|         0.0|     30.0|
|/XAabCrXsPJpWAYH7...|           1|        99|             0|         1|         0.0|     30.0|
|0w5uJIBRwgQCiwllN...|           1|        99|             0|         1|         0.0|     30.0|
|29bZgi/OGobH7zhPg...|           2|       298|             1|         1|         0.0|     30.0|
+--------------------+------------+----------+--------------+----------+------------+---------+
only showing top 5 rows


combining everything into one customer table.

In [0]:
customers = train.join(members.select("msno", "city", "age", "gender", "registered_via", "signup_date"), on="msno", how="left")
customers = customers.join(payments, on="msno", how="left")
customers = customers.join(listening, on="msno", how="left")

In [0]:
customers = customers.fillna({"total_secs": 0, "active_days": 0, "unique_songs": 0})

In [0]:
print(customers.count())
customers.show(5)

970960
+--------------------+--------+----+----+-------+--------------+-----------+------------+----------+--------------+----------+------------+---------+------------------+-----------+------------+
|                msno|is_churn|city| age| gender|registered_via|signup_date|num_payments|total_paid|ever_cancelled|auto_renew|avg_discount|plan_days|        total_secs|active_days|unique_songs|
+--------------------+--------+----+----+-------+--------------+-----------+------------+----------+--------------+----------+------------+---------+------------------+-----------+------------+
|ZwIu/ExmWZCB9wr1w...|       0|  13|  21| female|             9| 2014-01-12|           2|       298|             0|         1|         0.0|     30.0|        812773.701|         31|        2555|
|G7tsia3XiyYS/vdK0...|       0|   1|NULL|unknown|             7| 2015-07-12|           1|        99|             0|         1|         0.0|     30.0| 81013.16500000001|         30|         407|
|m+d7f4BMIiBqvlFKe...| 

how long they’ve been a customer

In [0]:
customers = customers.withColumn("tenure_days", F.datediff(F.to_date(F.lit("2017-03-31")), col("signup_date")))

save the clean table in Databricks

In [0]:
customers.write.mode("overwrite").saveAsTable("workspace.default.customers_clean")

save a copy as a CSV file

In [0]:
customers.coalesce(1).write.mode("overwrite").option("header", True).csv(raw + "/customers_clean")